# Corpus totals (72 lists)

In [1]:
import glob
import os

import pandas as pd

BUILD_THREADS_DIR = '../../build_threads_output'

files = sorted(glob.glob(os.path.join(BUILD_THREADS_DIR, 'list_data_*.parquet')))

per_list = pd.DataFrame(
    [
        {
            'list': os.path.basename(path).removeprefix('list_data_').removesuffix('.parquet'),
            'threads': len(df),
            'emails': int(df['n_messages'].sum()),
        }
        for path, df in ((path, pd.read_parquet(path, columns=['n_messages'])) for path in files)
    ]
)

overall = pd.DataFrame({
    'metric': ['total_lists', 'total_emails', 'total_threads'],
    'value': [len(per_list), per_list['emails'].sum(), per_list['threads'].sum()],
})
overall

,metric,value
0,total_lists,72
1,total_emails,3521123
2,total_threads,670211


# Per list

In [2]:
per_list.sort_values('emails', ascending=False).reset_index(drop=True)

,list,threads,emails
0,netdev,216279,1229125
1,linux-fsdevel,43907,352187
2,bpf,23203,188884
3,amd,27141,146423
4,linux-kselftest,15948,123143
...,...,...,...
67,opae,40,70
68,lch,13,41
69,printing-users,14,40
70,mailbox,7,26


# Pre-filter regex terms

In [3]:
import sys

sys.path.insert(0, '../..')
from pre_filter.pre_filter_threads import CANDIDATE_RE


def top_level_alternatives(pattern):
    body = pattern.removeprefix(r'\b(?:').removesuffix(r')\b')
    parts, depth, current = [], 0, ''
    for char in body:
        if char == '(':
            depth += 1
        elif char == ')':
            depth -= 1
        if char == '|' and depth == 0:
            parts.append(current)
            current = ''
        else:
            current += char
    return parts + [current]


pd.DataFrame({'pattern': top_level_alternatives(CANDIDATE_RE.pattern)})

,pattern
0,duplicat[a-z]*
1,dedup[a-z]*
2,redundant[a-z]*
3,repeated
4,copy[-_\s]?past(?:e|ed|ing)


# Pre-filter results

In [4]:
PRE_FILTER_DIR = '../../pre_filter/pre_filter_output'

pre_filter_files = sorted(glob.glob(os.path.join(PRE_FILTER_DIR, 'list_data_*.parquet')))

candidates = pd.DataFrame(
    [
        {
            'list': os.path.basename(path).removeprefix('list_data_').removesuffix('.parquet'),
            'candidate_threads': len(df),
            'candidate_emails': int(df['n_messages'].sum()),
        }
        for path, df in ((path, pd.read_parquet(path, columns=['n_messages'])) for path in pre_filter_files)
    ]
)

pre_filter = per_list.merge(candidates, on='list')
pre_filter['selection_rate'] = (pre_filter['candidate_threads'] / pre_filter['threads']).round(3)

pd.DataFrame(
    {
        'metric': ['total_candidate_emails', 'total_candidate_threads', 'selection_rate'],
        'value': [
            pre_filter['candidate_emails'].sum(),
            pre_filter['candidate_threads'].sum(),
            round(pre_filter['candidate_threads'].sum() / pre_filter['threads'].sum(), 3),
        ],
    },
    dtype=object,
)

,metric,value
0,total_candidate_emails,895474
1,total_candidate_threads,66245
2,selection_rate,0.099


# Pre-filter results per list

In [5]:
pre_filter[['list', 'emails', 'threads', 'candidate_emails', 'candidate_threads', 'selection_rate']].sort_values(
    'candidate_threads', ascending=False
).reset_index(drop=True)

,list,emails,threads,candidate_emails,candidate_threads,selection_rate
0,netdev,1229125,216279,276351,22215,0.103
1,linux-fsdevel,352187,43907,131515,7204,0.164
2,linux-next,86362,35196,13824,4906,0.139
3,bpf,188884,23203,61533,3903,0.168
4,amd,146423,27141,34939,2548,0.094
...,...,...,...,...,...,...
67,printing-users,40,14,16,1,0.071
68,lch,41,13,0,0,0.000
69,mailbox,26,7,0,0,0.000
70,lvfs-general,14,9,0,0,0.000


# Threads per matched term

In [6]:
matched_terms = pd.concat(
    [pd.read_parquet(path, columns=['matched_terms'])['matched_terms'] for path in pre_filter_files],
    ignore_index=True,
)

per_term_threads = matched_terms.str.split(',').explode().value_counts().rename('threads').to_frame()
per_term_threads['share_of_candidates'] = (per_term_threads['threads'] / len(matched_terms)).round(3)
per_term_threads

,threads,share_of_candidates
matched_terms,,
duplicate,25029,0.378
redundant,22958,0.347
duplicated,10209,0.154
duplication,7950,0.120
repeated,5684,0.086
...,...,...
duplicatign,1,0.000
dedupplicate,1,0.000
duplicateerror,1,0.000


# True and false positive rate per matched term

In [7]:
CONSOLIDATED = [
    'consolidated/sample_review_v2_consolidated.csv',
    'consolidated/sample_review_usp_v2_consolidated.csv',
]

reviewed = pd.concat(
    [pd.read_csv(path, dtype=str, keep_default_na=False) for path in CONSOLIDATED],
    ignore_index=True,
)
# The stored matched_terms come from the pre_filter version the sample was
# drawn with; recompute them with the current one so the table reflects what
# the pre-filter actually picks up now. Threads it no longer matches drop out.
from pre_filter.pre_filter_threads import matches, replace_quotes

reviewed['term'] = reviewed['thread_content'].map(lambda content: matches(replace_quotes(content)))
matched = reviewed[reviewed['term'].str.len() > 0].copy()

per_term = (
    matched.explode('term')
    .groupby('term')['final_decision']
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={'yes': 'true_positives', 'no': 'false_positives'})
)
per_term['threads'] = per_term['true_positives'] + per_term['false_positives']

summary = pd.DataFrame({
    'threads': per_term['threads'],
    'true_positives (%)': (100 * per_term['true_positives'] / per_term['threads']).round(1),
    'false_positives (%)': (100 * per_term['false_positives'] / per_term['threads']).round(1),
})

summary.loc[per_term.sort_values(['true_positives', 'threads'], ascending=False).index].rename_axis('term').reset_index()

,term,threads,true_positives (%),false_positives (%)
0,duplication,40,92.5,7.5
1,duplicate,52,34.6,65.4
2,redundant,70,22.9,77.1
3,duplicated,27,55.6,44.4
4,deduplicate,15,93.3,6.7
5,repeated,21,47.6,52.4
6,duplicating,11,90.9,9.1
7,common-helper,8,100.0,0.0
8,copy-pasted,6,100.0,0.0
9,duplications,6,100.0,0.0
